# Retrieval Chain With LCEL

This notebook builds and runs a retrieval workflow using LangChain Expression Language (LCEL), then compares the structure to a simpler baseline approach.

## Imports and Dependencies

Load core Python utilities and LangChain modules required for embeddings, prompts, runnables, and vector store integration.

In [18]:
from pathlib import Path
import os
from dotenv import load_dotenv
from operator import itemgetter

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


## Project and Environment Setup

Resolve the project directory and load environment variables from `.env` so API keys and index configuration are available at runtime.

In [19]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'ingestion.py').exists():
    PROJECT_DIR = Path('/Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer')

load_dotenv(PROJECT_DIR / '.env')
print('Loaded .env from:', PROJECT_DIR / '.env')
print('INDEX_NAME:', os.getenv('INDEX_NAME'))

Loaded .env from: /Users/abtinzandi/Documents/projects/LLM-GANs-DP/LLM-GANs-Projects/LangChain/udemy_course/Medium_analyzer/.env
INDEX_NAME: medium-analyer


## User Query

Define the question that will be sent through the retrieval pipeline.

In [20]:
QUERY = "what is Pinecone in machine learning?"

## Baseline Model Objects

Initialize default embeddings and LLM instances for quick validation before explicit credential wiring.

In [21]:
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI()

## Credential and Index Validation

Validate required environment variables and initialize Pinecone connectivity settings.

In [23]:
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables.")

index_name = os.getenv('INDEX_NAME')
if not index_name:
    raise ValueError("INDEX_NAME not found in environment variables.")

print('Using OpenAI API Key:', openai_api_key[:6] + '...')
print('Using Pinecone Index Name:', index_name)

Using OpenAI API Key: sk-pro...
Using Pinecone Index Name: medium-analyer


## Vector Store Initialization

Create the Pinecone-backed vector store and confirm connection to the target index.

In [24]:
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embeddings
)

print('Connected to Pinecone index:', index_name)

Connected to Pinecone index: medium-analyer


## Retriever Construction

Configure a retriever with `k=3` to fetch the top matching chunks for each query.

In [25]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print('Retriever created with k=3')

Retriever created with k=3


## Prompt + LCEL Pipeline (Inline)

Define a prompt template and assemble an LCEL chain that injects retrieved context before calling the chat model.

In [26]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt_template = ChatPromptTemplate.from_template(
   template= """Answer the question based only on the following context:

{context}

Question: {question}

Provide a detailed answer:"""
)

prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}\n\nProvide a detailed answer:'), additional_kwargs={})])

## Helper Utilities

Add small formatting helpers to convert retrieved documents into prompt-ready context text.

In [27]:
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

## Reusable LCEL Chain Builder

Wrap the LCEL construction into a function so the retrieval chain can be recreated and invoked consistently.

Advantages over non-LCEL approach:

    - Declarative and composable: Easy to chain operations with pipe operator (|)
    - Built-in streaming: chain.stream() works out of the box
    - Built-in async: chain.ainvoke() and chain.astream() available
    - Batch processing: chain.batch() for multiple inputs
    - Type safety: Better integration with LangChain's type system
    - Less code: More concise and readable
    - Reusable: Chain can be saved, shared, and composed with other chains
    - Better debugging: LangChain provides better observability tools

In [28]:
def create_retrieval_chain_with_lcel():

    chain = (
        RunnablePassthrough.assign(
            context=itemgetter("question") | retriever | format_docs
        ) 
        | prompt_template
        | llm
        | StrOutputParser()
    )

    return chain

## Visualize the Chain

In [33]:
def visualize_chain(chain, log=True, save_png=False, png_path='retrieval_chain_graph.png'):
    """Visualize an LCEL chain as ASCII, Mermaid text, and optional PNG."""
    graph = chain.get_graph()

    if log:
        print('=' * 70)
        print('LCEL CHAIN (ASCII)')
        print('=' * 70)
        try:
            graph.print_ascii()
        except Exception as exc:
            print(f'ASCII render not available: {exc}')

        print('\n' + '=' * 70)
        print('LCEL CHAIN (MERMAID)')
        print('=' * 70)
        try:
            mermaid = graph.draw_mermaid()
            print(mermaid)
        except Exception as exc:
            print(f'Mermaid render not available: {exc}')

    if save_png:
        try:
            graph.draw_png(png_path)
            print(f'\nGraph image saved to: {png_path}')
        except Exception as exc:
            print(f'PNG export not available: {exc}')


chain = create_retrieval_chain_with_lcel()
visualize_chain(chain, save_png=False)


LCEL CHAIN (ASCII)
ASCII render not available: Install grandalf to draw graphs: `pip install grandalf`.

LCEL CHAIN (MERMAID)
---
config:
  flowchart:
    curve: linear
---
graph TD;
	Parallel_context_Input([Parallel<context>Input]):::first
	Parallel_context_Output(Parallel<context>Output)
	Lambda(Lambda)
	VectorStoreRetriever(VectorStoreRetriever)
	format_docs(format_docs)
	Passthrough(Passthrough)
	ChatPromptTemplate(ChatPromptTemplate)
	ChatOpenAI(ChatOpenAI)
	StrOutputParser(StrOutputParser)
	StrOutputParserOutput([StrOutputParserOutput]):::last
	Lambda --> VectorStoreRetriever;
	VectorStoreRetriever --> format_docs;
	Parallel_context_Input --> Lambda;
	format_docs --> Parallel_context_Output;
	Parallel_context_Input --> Passthrough;
	Passthrough --> Parallel_context_Output;
	Parallel_context_Output --> ChatPromptTemplate;
	ChatPromptTemplate --> ChatOpenAI;
	StrOutputParser --> StrOutputParserOutput;
	ChatOpenAI --> StrOutputParser;
	classDef default fill:#f2f0ff,line-height:1.2
	

## Execution and Comparison Notes

Run the LCEL implementation, print results, and summarize why the LCEL approach is easier to maintain and extend.

In [35]:

print("\n" + "=" * 70)
print("IMPLEMENTATION 2: With LCEL - Better Approach")
print("=" * 70)
print("Why LCEL is better:")
print("- More concise and declarative")
print("- Built-in streaming: chain.stream()")
print("- Built-in async: chain.ainvoke()")
print("- Easy to compose with other chains")
print("- Better for production use")
print("=" * 70)

chain_with_lcel = create_retrieval_chain_with_lcel()
result_with_lcel = chain_with_lcel.invoke({"question": QUERY})
print("\nAnswer:")
print(result_with_lcel)


IMPLEMENTATION 2: With LCEL - Better Approach
Why LCEL is better:
- More concise and declarative
- Built-in streaming: chain.stream()
- Built-in async: chain.ainvoke()
- Easy to compose with other chains
- Better for production use

Answer:
Pinecone is a platform designed for fast and scalable retrieval of similar data points based on their vector representations in machine learning. It is capable of handling large-scale ML applications with millions or billions of data points and provides infrastructure management and maintenance to its users. Pinecone can handle high query throughput and low latency search, making it efficient for real-time data retrieval. Additionally, Pinecone is a secure platform that meets the security needs of businesses and organizations.

One of the key features of Pinecone is its user-friendly design and accessibility through a simple API for storing and retrieving vector data. This makes it easy to integrate Pinecone into existing ML workflows. Pinecone als